# 02. Entailment dan Inference


## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel mana pun yang lain.

Sel ini memasang dependensi yang diperlukan, mencari folder yang berisi
`logic.py` dan `utils.py`, lalu mengimpornya. Kalau notebook dibuka lewat Google
Colab, repo akan di-clone otomatis. Tidak ada yang perlu diubah di sini.

Environment sudah siap kalau baris terakhir output mencetak
`Check       : tt_entails(P & Q, Q) = True`.

In [ ]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

---
# 2.1 Terminologi Dasar

**Slide 17**

## Penjelasan

Yang perlu masuk, urut sesuai slide:

- **Syntax**: aturan pembentukan sentence yang valid dalam suatu representation
  language. Contoh dari slide: dalam aritmatika, `x + y = 4` adalah sentence yang
  well-formed, sedangkan `x3y+ =` tidak.
- **Semantics**: makna atau nilai kebenaran sentence. Contoh dari slide:
  `x + y = 4` bernilai benar di dunia di mana x adalah 2 dan y adalah 2, tapi
  salah di dunia di mana x adalah 1 dan y adalah 1.
- **Model**: kalau sentence $\alpha$ bernilai benar di model $m$, maka $m$
  memenuhi $\alpha$, atau $m$ adalah model dari $\alpha$.
- **Notasi $M(\alpha)$**: himpunan semua model dari $\alpha$.
- **Logical entailment**: relasi antar sentence, ditulis
  $\alpha \vDash \beta$, artinya $\beta$ mengikuti secara logis dari $\alpha$.

Yang paling penting dan paling sering keliru: bedanya "model" dalam arti logika
dengan "model" dalam arti machine learning. Sebutkan sekali di awal supaya tidak
tertukar sepanjang materi.

Definisi formal entailment lewat himpunan model, yaitu
$\alpha \vDash \beta$ jika dan hanya jika $M(\alpha) \subseteq M(\beta)$,
perlu ditulis eksplisit di sini karena akan dipakai terus sampai Notebook 04.

## Contoh penerapan

Ambil contoh aritmatika dari slide, bukan contoh logika, supaya konsep syntax
dan semantics terpisah jelas dari materi berikutnya. Buat beberapa nilai x dan y,
lalu tunjukkan model mana yang memenuhi `x + y = 4` dan mana yang tidak.

Cukup pakai loop Python biasa.

---
# 2.2 Logical Entailment Sample: Membangun Ruang Model

**Slide 18 dan 19**

## Penjelasan

Yang perlu masuk:

- Situasinya: agent tidak mendeteksi apa pun di [1,1] dan merasakan breeze di
  [2,1].
- Percept ini digabung dengan pengetahuan agent tentang aturan Wumpus World
  membentuk KB.
- Agent ingin tahu apakah [1,2], [2,2], dan [3,1] berisi pit.
- Tiap kotak bisa berisi pit atau tidak, jadi ada $2^3 = 8$ possible model.
- KB bernilai salah di model yang bertentangan dengan apa yang diketahui agent.
  Contoh dari slide: KB salah di model mana pun yang menempatkan pit di [1,2],
  karena tidak ada breeze di [1,1].
- Hasilnya, ada tepat 3 model di mana KB bernilai benar.

Perlu ditekankan: yang dienumerasi cuma tiga kotak yang belum diketahui, bukan
seluruh 16 kotak. Kotak yang sudah dikunjungi tidak divariasikan karena agent
masih hidup.

## Contoh penerapan

Enumerasi kedelapan model pakai `itertools.product`. Tiap model direpresentasikan
sebagai dict dengan key string, misalnya `{'P12': False, 'P22': False, 'P31': True}`.

Tulis fungsi yang mengecek apakah sebuah model konsisten dengan KB, yaitu tidak
ada breeze di [1,1] dan ada breeze di [2,1]. Aturannya diimplementasi sebagai
kondisi Python biasa, belum sebagai sentence logika.

Tampilkan hasilnya sebagai tabel 8 baris dengan kolom terakhir menandai apakah KB
bernilai benar. Verifikasi bahwa jumlah barisnya benar 3, sesuai slide.

---
# 2.3 Dua Kesimpulan: Alpha 1 dan Alpha 2

**Slide 20 dan 21, Figure 7.5**

## Penjelasan

Yang perlu masuk:

- $\alpha_1$ = "tidak ada pit di [1,2]". Di setiap model di mana KB benar,
  $\alpha_1$ juga benar. Jadi $KB \vDash \alpha_1$.
- $\alpha_2$ = "tidak ada pit di [2,2]". Ada model di mana KB benar tapi
  $\alpha_2$ salah. Jadi $KB \nvDash \alpha_2$.
- Hubungkan dengan definisi himpunan: $M(KB) \subseteq M(\alpha_1)$ tapi
  $M(KB) \not\subseteq M(\alpha_2)$. Ini persis yang digambarkan Figure 7.5.

Bagian yang paling sering disalahpahami dan wajib dibahas eksplisit:
$KB \nvDash \alpha_2$ **tidak** berarti $\alpha_2$ salah. Artinya cuma agent
belum punya cukup informasi untuk memastikan. Cek juga bahwa
$KB \nvDash \neg\alpha_2$. Dua-duanya tidak entailed sekaligus, dan itu wajar.

Tampilkan Figure 7.5 di sini, karena diagram himpunannya jauh lebih jelas
daripada dijelaskan dengan kata-kata.

## Contoh penerapan

Lanjutkan tabel dari sub-topik 2.2, tambahkan dua kolom untuk $\alpha_1$ dan
$\alpha_2$. Lalu cek dengan kode:

- Apakah $\alpha_1$ benar di semua baris yang KB-nya benar?
- Apakah $\alpha_2$ benar di semua baris yang KB-nya benar?
- Apakah negasi $\alpha_2$ benar di semua baris yang KB-nya benar?

Ketiga jawabannya berturut-turut True, False, False. Bahas kenapa dua yang
terakhir sama-sama False.

---
# 2.4 Logical Inference

**Slide 22 dan 23, Figure 7.6**

## Penjelasan

Yang perlu masuk:

- Entailment bisa dipakai untuk menurunkan kesimpulan, dan proses itu disebut
  logical inference.
- Algoritma yang dipakai di sub-topik sebelumnya namanya **model checking**,
  karena dia mengenumerasi semua possible model untuk memastikan
  $M(KB) \subseteq M(\alpha)$.
- **Sound** atau truth preserving: algoritma inference yang hanya menurunkan
  sentence yang memang entailed. Model checking bersifat sound.
- **Complete**: algoritma yang bisa menurunkan semua sentence yang entailed.
- Figure 7.6: sentence adalah konfigurasi fisik dari agent, dan reasoning adalah
  proses membentuk konfigurasi baru dari yang lama. Logical reasoning menjamin
  konfigurasi baru itu merepresentasikan aspek dunia yang memang mengikuti dari
  konfigurasi lama.

Jelaskan konsekuensi praktis dari sound dan complete. Algoritma yang sound tapi
tidak complete artinya apa, dan sebaliknya. Mana yang lebih berbahaya kalau
dilanggar.

Tutup dengan menyebut bahwa model checking punya masalah: jumlah modelnya
$2^n$. Ini jembatan ke Notebook 04, tapi jangan dibahas detail di sini.

## Contoh penerapan

Bungkus pengecekan dari sub-topik 2.3 menjadi satu fungsi `entails(kb, alpha)`
yang menerima dua fungsi predikat dan mengembalikan True atau False. Ini versi
manual dari `tt_entails` yang akan dipakai di Notebook 04.

Uji dengan beberapa pasangan sederhana, misalnya KB berisi dua fakta dan alpha
salah satunya. Jangan pakai Wumpus lagi di sini supaya tidak berulang.

---
# Latihan Soal

Isi bagian ini dengan minimal 3 soal. Susun dari yang paling ringan.

Format tiap soal: pernyataan soal, cell kosong untuk jawaban, lalu pembahasan
yang dibungkus `<details>`.

## Soal 1

Tingkat pemahaman. Usul arah soal: jelaskan bedanya "alpha salah di semua model
KB" dengan "alpha tidak benar di semua model KB". Kaitkan dengan kasus
$\alpha_2$ di sub-topik 2.3.

## Soal 2

Tingkat penerapan. Usul arah soal: buktikan atau bantah $P \vDash P \lor Q$
dengan membuat tabel semua model. Minta pembaca menulis kodenya, bukan cuma
menjawab ya atau tidak.

## Soal 3

Tingkat analisis. Usul arah soal: beri satu contoh KB dan alpha di mana
$KB \nvDash \alpha$ dan sekaligus $KB \nvDash \neg\alpha$. Jelaskan apa
artinya kondisi itu bagi agent yang harus mengambil keputusan.

## Soal 4 (opsional)

Ruang untuk soal tambahan. Usul arah: kalau sebuah algoritma inference sound tapi
tidak complete, apa yang bisa terjadi pada agent Wumpus? Bagaimana kalau
sebaliknya, complete tapi tidak sound? Mana yang lebih berbahaya.

Hapus kalau tidak dipakai.